# **MECS6616 Spring 2026 - Assignment 3: Diffusion Policy and Reinforcement Learning**

***IMPORTANT:***
- **Before starting, make sure to read the [Assignment Instructions](https://courseworks2.columbia.edu/courses/237884/pages/assignment-instructions) page on Courseworks to understand the workflow and submission requirements for this project.**

***FOR Assignment 3!!!***
- Apart from the link to your notebook, you are also required to submit your chosen model checkpoint `.pth` files to Coursework. You will have one file for each part.
- Your part 2 files should be named `diffusion_policy.pth` and `rl_policy.zip`.
- You should put the link to your notebook in the "Comment" section of your submission.

# **Assignment Setup & Imports (do NOT change)**

***IMPORTANT:***
- Do NOT change this "*Assignment & Imports*" section
- Do NOT install any other dependencies or a different version of an already provided package. You may, however, import other packages

1. Install required Python packages (ignore the **ERROR** for "shapely" in ouput)

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# Installing required packages
!pip install -q --upgrade pip
!pip install -q scikit-video==1.1.11
!pip install -q zarr==2.12.0
!pip install -q numcodecs==0.15.1
!pip install -q pygame==2.6.1
!pip install -q pymunk==6.2.1
!pip install -q gym==0.26.2
!pip install -q shapely==1.8.4
!pip install -q diffusers==0.32.2
!pip install -q numpy==2.0.2
!pip install -q scikit-image==0.23.2
!pip install -q wheel==0.38.4
!pip install -q gym stable-baselines3
!pip install -q shimmy==1.3.0

2. Install dm_control

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

!apt-get install -qq -y libosmesa6-dev > /dev/null 2>&1
%pip install -q dm_control imageio imageio-ffmpeg

3. Clone repo from Github

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# After running this cell, the folder 'mece6616_sp25_project4' will show up in the file explorer on the left (click on the folder icon if it's not open)
# It may take a few seconds to appear
!git clone https://github.com/roamlab/mece6616_sp25_project4.git
!mv /content/mece6616_sp25_project4/* /content/
!rm -rf mece6616_sp25_project4

# Import required packages

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

import os
os.environ['MUJOCO_GL'] = 'osmesa'
# IMPORTANT lines above must be run consecutively

import io
import base64
import copy
import math
import time
import collections
from collections import OrderedDict
from typing import Union

import numpy as np
import imageio
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import cv2
import gdown
import zipfile
import pickle

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils import spectral_norm
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, random_split

import pymunk
import gymnasium as gym
from gymnasium import spaces

from dm_control import suite
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env.subproc_vec_env import SubprocVecEnv
from stable_baselines3.common.vec_env.vec_monitor import VecMonitor

from tqdm.notebook import tqdm
from IPython.display import HTML, Video, clear_output
from skvideo.io import vwrite

from PushT import PushTEnv
from dataset import PushTStateDataset, normalize_data, unnormalize_data
from ema import EMAModel
from scheduler import get_cosine_schedule_with_warmup
from util import show_video

# ================================
# Environment Setup
# ================================

device = torch.device('cpu')

# ================================
# Activation Functions
# ================================

activation_dict = nn.ModuleDict({
    "ReLU":      nn.ReLU(),
    "ELU":       nn.ELU(),
    "GELU":      nn.GELU(),
    "Tanh":      nn.Tanh(),
    "Mish":      nn.Mish(),
    "Identity":  nn.Identity(),
    "Softplus":  nn.Softplus(),
    "LeakyReLU": nn.LeakyReLU(),
    "Sigmoid":   nn.Sigmoid(),
})

# ================================
# Visualization Utilities
# ================================

def show_video(frames, fps=30):
    """Display frames as an inline MP4 video."""
    buf = io.BytesIO()
    imageio.mimsave(buf, frames, format='mp4', fps=fps)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    return HTML(
        f'<video controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    )


def visualize_log(ticks, rewards, frames, qposs, qvels):
    """Plot reward, joint positions, and joint velocities over time."""
    ticks = [t * 4 for t in ticks]
    qposs = np.asarray(qposs)
    qvels = np.asarray(qvels)

    fig, ax = plt.subplots(3, 3, sharex=True, figsize=(12, 8))

    # Row 0: Reward
    ax[0, 0].plot(ticks, rewards)
    ax[0, 0].set_ylabel('reward')
    ax[0, 0].set_title('Reward')
    ax[0, 1].axis('off')
    ax[0, 2].axis('off')

    # Row 1: Positions
    for col, (label, title) in enumerate(zip(
        ['x position', 'z position', 'pitch position'],
        ['X Position', 'Z Position', 'Pitch Position']
    )):
        ax[1, col].plot(ticks, qposs[:, col])
        ax[1, col].set_ylabel(label)
        ax[1, col].set_title(title)

    # Row 2: Velocities
    for col, (label, title) in enumerate(zip(
        ['x velocity', 'z velocity', 'pitch velocity'],
        ['X Velocity', 'Z Velocity', 'Pitch Velocity']
    )):
        ax[2, col].plot(ticks, qvels[:, col])
        ax[2, col].set_ylabel(label)
        ax[2, col].set_xlabel('time')
        ax[2, col].set_title(title)

    plt.tight_layout()
    plt.show()

# **Part1. Diffusion Policy for PushT**


In this project, you will implement various agents to perform Behavioral Cloning, including a simple **MLP** agent, a **Conditional Variational Autoencoder (CVAE)** agent, and a **Diffusion Policy** agent.

The assignment is based on the state-based Push-T environment. In this environment, the objective is to push a T-shaped block from various starting poses to a fixed goal pose using a 2D locomotion robot.

The Push-T environment features an observation space of length 5 and an action space of length 2:

*   **Obs:** [agent_pos **(2)**, ob_pos **(2)**, obj_rotation **(1)**]
*   **Action:** $[x, y]$

Here, $x$ and $y$ represent the predicted absolute position of the agent for the next move. And the action is executed using a PD controller.


<div>
<img src="https://github.com/roamlab/mece6616_sp25_project4/blob/main/imgs/pusht.png?raw=true" width="300"/>
</div>

We have preconfigured the model hyperparameters for you, so you will not need to spend time tuning them. We recommend **NOT** modifying these hyperparameters and instead focusing on implementing and refining the training and inference scripts.

You can monitor the loss curves during training and compare them to the expected trends we provided. In this case, you could stop training earlier if it becomes clear that their agent is not converging as expected.

As you successfully complete these scripts for each algorithm, you will observe a consistent improvement in their performance on this manipulation task.




## **Download Datasets for Behavioral Cloning**

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# download demonstration data from Google Drive
dataset_path = "pusht_cchi_v7_replay.zarr.zip"
if not os.path.isfile(dataset_path):
    id = "1KY1InLurpMvJDRb14L9NlXT_fEsCvVUq&confirm=t"
    gdown.download(id=id, output=dataset_path, quiet=False)

# Extract dataset
extract_path = "/content/pusht_cchi_v7_replay.zarr"
if not os.path.exists(extract_path):
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

## **Push-T Task Environment**

Below are some useful APIs provided by the Push-T environment used in this assignment. You may use them for debugging purposes.

1.   **seed(seed):** Set the random seed in the environment
  - Args:
      - seed: the seed to set (int)
  - Returns: None

2.   **reset():** Reset the environment with randomly sampled initial state.
  - Args: None
  - Returns:
      - obs: initial observation (np.array)
      - info: initial hidden parameter values of the environment (dict)

3.   **render(mode):** Render the next frame
  - Args:
      - mode: default as 'rgb_array'
  - Returns: None

4.   **step(action):** Step the environment with the given action.
  - Args:
      - action: action to take (np.array)
  - Returns:          
      - obs: next observation after taking the action (np.array)
      - reward: reward after taking the action (float)
      - terminated: whether the episode is terminated (bool)
      - truncated: whether the episode is truncated (bool)
      - info: hidden parameter values after taking the action (dict)

5.   **set_obs(obs):** Set the certain state to the environment.
  - Args:
      - obs: state to set (torch.Tensor)
  - Returns: None

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# Always ignore `.git` and `.cache/huggingface` folders in commits
IGNORE_GIT_FOLDER_PATTERNS = [
    ".git",
    ".git/*",
    "*/.git",
    "**/.git/**",
    ".cache/huggingface",
    ".cache/huggingface/*",
    "*/.cache/huggingface",
    "**/.cache/huggingface/**",
]

# 0. create env object
env = PushTEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, IGNORE_GIT_FOLDER_PATTERNS = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("Obs: ", repr(obs))
    print("Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]")
    print("Action: ", repr(action))
    print("Action:   [target_agent_x, target_agent_y]")

## **Receding-Horizon Control**
Similar to the MPC framework, we apply *receding-horizon control* in this assignment as follows: At each time step $t$, the policy takes the most recent $T_o$ steps of observation data $O_t$ as input, and predicts $T_a$ steps of future actions $A_t$. This stratetgy is illustrated in the Diffusion Policy paper as follows:

<div>
<img src="https://github.com/roamlab/mece6616_sp25_project4/blob/main/imgs/receding.png?raw=true" width="300"/>
</div>

**Important**: In the Push-T task implementation we use, the number of actions predicted by the agent (16) differs from the number of actions actually executed during inference (8).


## **Create Dataloader**

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=extract_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['obs'].shape:", batch['obs'].shape)
print("batch['action'].shape", batch['action'].shape)

## **Visualize Demo**

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

# keep a queue of last 2 steps of observations
# save visualization and rewards
demo_imgs = [env.render(mode='rgb_array')]

# for i in range(batch['obs'].shape[0]):
for i in range(50):
  naction = batch['action'][i]
  nobs = batch['obs'][i]

  obs = unnormalize_data(nobs, stats=stats['obs'])

  # get first observation
  env.set_obs(obs[0])


  # execute action_horizon number of steps
  # without replanning
  for j in range(naction.shape[0]):
      # fetch next action
      action = unnormalize_data(naction[j], stats=stats['action'])
      # stepping env
      obs, reward, done, _, info = env.step(action)
      # saved rendered image
      demo_imgs.append(env.render(mode='rgb_array'))

# create video
vwrite('demo_vis.mp4', demo_imgs, inputdict={'-r': '10'})  # Set frame rate to 10 fps
Video('demo_vis.mp4', embed=True, width=256, height=256)

# **Part 1. Behavioral Cloning with Diffusion Policy**

In this section, we will focus on implementing the Denoising Diffusion Probabilistic Model (DDPM) and perform Behavioral Cloning based on the Diffusion Policy, as introduced in the guest lecture by Zhanpeng He.

You will be provided with a pre-implemented U-Net architecture and fixed hyperparameters that serve as the noise prediction network.

Your task will be to implement the key components of DDPM:
1. Cosine noise schedule
2. Forward diffusion process
3. Backward denoising process

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb


class Downsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv1d(dim, dim, 3, 2, 1)

    def forward(self, x):
        return self.conv(x)

class Upsample1d(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.ConvTranspose1d(dim, dim, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


class Conv1dBlock(nn.Module):
    '''
        Conv1d --> GroupNorm --> Mish
    '''

    def __init__(self, inp_channels, out_channels, kernel_size, n_groups=8):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv1d(inp_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.GroupNorm(n_groups, out_channels),
            nn.Mish(),
        )

    def forward(self, x):
        return self.block(x)


class ConditionalResidualBlock1D(nn.Module):
    def __init__(self,
            in_channels,
            out_channels,
            cond_dim,
            kernel_size=3,
            n_groups=8):
        super().__init__()

        self.blocks = nn.ModuleList([
            Conv1dBlock(in_channels, out_channels, kernel_size, n_groups=n_groups),
            Conv1dBlock(out_channels, out_channels, kernel_size, n_groups=n_groups),
        ])

        # FiLM modulation https://arxiv.org/abs/1709.07871
        # predicts per-channel scale and bias
        cond_channels = out_channels * 2
        self.out_channels = out_channels
        self.cond_encoder = nn.Sequential(
            nn.Mish(),
            nn.Linear(cond_dim, cond_channels),
            nn.Unflatten(-1, (-1, 1))
        )

        # make sure dimensions compatible
        self.residual_conv = nn.Conv1d(in_channels, out_channels, 1) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x, cond):
        '''
            x : [ batch_size x in_channels x horizon ]
            cond : [ batch_size x cond_dim]

            returns:
            out : [ batch_size x out_channels x horizon ]
        '''
        out = self.blocks[0](x)
        embed = self.cond_encoder(cond)

        embed = embed.reshape(
            embed.shape[0], 2, self.out_channels, 1)
        scale = embed[:,0,...]
        bias = embed[:,1,...]
        out = scale * out + bias

        out = self.blocks[1](out)
        out = out + self.residual_conv(x)
        return out


class ConditionalUnet1D(nn.Module):
    def __init__(self,
        input_dim,
        global_cond_dim,
        diffusion_step_embed_dim=256,
        down_dims=[256,512,1024],
        kernel_size=5,
        n_groups=8
        ):
        """
        input_dim: Dim of actions.
        global_cond_dim: Dim of global conditioning applied with FiLM
          in addition to diffusion step embedding. This is usually obs_horizon * obs_dim
        diffusion_step_embed_dim: Size of positional encoding for diffusion iteration k
        down_dims: Channel size for each UNet level.
          The length of this array determines numebr of levels.
        kernel_size: Conv kernel size
        n_groups: Number of groups for GroupNorm
        """

        super().__init__()
        all_dims = [input_dim] + list(down_dims)
        start_dim = down_dims[0]

        dsed = diffusion_step_embed_dim
        diffusion_step_encoder = nn.Sequential(
            SinusoidalPosEmb(dsed),
            nn.Linear(dsed, dsed * 4),
            nn.Mish(),
            nn.Linear(dsed * 4, dsed),
        )
        cond_dim = dsed + global_cond_dim

        in_out = list(zip(all_dims[:-1], all_dims[1:]))
        mid_dim = all_dims[-1]
        self.mid_modules = nn.ModuleList([
            # 1D CNN layer
            ConditionalResidualBlock1D(
                mid_dim, mid_dim, cond_dim=cond_dim,
                kernel_size=kernel_size, n_groups=n_groups
            ),
        ])

        down_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(in_out):
            is_last = ind >= (len(in_out) - 1)
            down_modules.append(nn.ModuleList([
                # 1D CNN layer
                ConditionalResidualBlock1D(
                    dim_in, dim_out, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                # Downsample layer
                Downsample1d(dim_out) if not is_last else nn.Identity()
            ]))

        up_modules = nn.ModuleList([])
        for ind, (dim_in, dim_out) in enumerate(reversed(in_out[1:])):
            is_last = ind >= (len(in_out) - 1)
            up_modules.append(nn.ModuleList([
                # 1D CNN layer
                ConditionalResidualBlock1D(
                    dim_out*2, dim_in, cond_dim=cond_dim,
                    kernel_size=kernel_size, n_groups=n_groups),
                # Upsample layer
                Upsample1d(dim_in) if not is_last else nn.Identity()
            ]))

        final_conv = nn.Sequential(
            # Conv1dBlock(start_dim, start_dim, kernel_size=kernel_size),
            nn.Conv1d(start_dim, input_dim, 1),
        )

        self.diffusion_step_encoder = diffusion_step_encoder
        self.up_modules = up_modules
        self.down_modules = down_modules
        self.final_conv = final_conv

        print("number of parameters: {:e}".format(
            sum(p.numel() for p in self.parameters()))
        )

    def forward(self,
            sample: torch.Tensor,
            timestep: Union[torch.Tensor, float, int],
            global_cond=None):
        """
            x: (B, T, input_dim)
            timestep: (B,) or int, diffusion step
            global_cond: (B, global_cond_dim)
            output: (B, T, input_dim)
        """
        # (B, T, C)
        sample = sample.moveaxis(-1, -2)
        # (B, C, T)

        # 1. time
        timesteps = timestep
        if not torch.is_tensor(timesteps):
            timesteps = torch.tensor([timesteps], dtype=torch.long, device=sample.device)
        elif torch.is_tensor(timesteps) and len(timesteps.shape) == 0:
            timesteps = timesteps[None].to(sample.device)
        timesteps = timesteps.expand(sample.shape[0])

        global_feature = self.diffusion_step_encoder(timesteps)

        if global_cond is not None:
            global_feature = torch.cat([
                global_feature, global_cond
            ], axis=-1)

        x = sample
        h = []
        for idx, (resnet, downsample) in enumerate(self.down_modules):
            x = resnet(x, global_feature)
            h.append(x)
            x = downsample(x)

        for mid_module in self.mid_modules:
            x = mid_module(x, global_feature)

        for idx, (resnet, upsample) in enumerate(self.up_modules):
            x = torch.cat((x, h.pop()), dim=1)
            x = resnet(x, global_feature)
            x = upsample(x)

        x = self.final_conv(x)

        # (B, C, T)
        x = x.moveaxis(-1, -2)
        # (B, T, C)
        return x

## **Denoising Diffusion Probabilistic Model (DDPM)**

### **Noise Schedule**
In Denoising Diffusion Probabilistic Models (DDPM), the beta schedule controls how much noise is added at each step of the forward diffusion process.

Recall that DDPM gradually adds Gaussian noise to a data sample $A_0$ over $K$ time steps to produce a noisy version $A_K$ that resembles pure noise.

At each step $k$, noise is added according to:
$$
A_k = \sqrt{\bar{\alpha}_k} \, A_0 + \sqrt{1 - \bar{\alpha}_k} \, \epsilon
$$

This is governed by a parameter $\beta_k \in (0, 1)$, which defines how much noise is added between steps $x_{k-1}$ and $x_k$. The relationship between $\alpha_k$ and $\beta_k$ is:
$$
\alpha_k = 1 - \beta_k, \quad \bar{\alpha}_k = \prod_{s=1}^k \alpha_s
$$


The beta schedule is the sequence $\beta_1, \beta_2, \dots, \beta_K$ that determines how quickly the signal is destroyed.

## **Q1.1: Implement the Cosine Beta Schedule.**

In this assignment, we focus on cosine beta schedule (as proposed in https://openreview.net/forum?id=-NEXDKk8gZ), which is the most popular choice in the diffusion model recently. The cosine beta schedule is a way to smoothly control the noise level which results in a slow start and accelerated noise toward the end — ideal for stable training and high-quality samples.


Instead of directly defining $\beta_k$, this schedule defines the cumulative product of alphas, denoted as $\bar{\alpha}_k$, using a cosine function:
$$
\bar{\alpha}_k = \frac{f(k)}{f(0)}, \quad f(k) = \cos^2\left( \frac{k/K + s}{1 + s} \cdot \frac{\pi}{2} \right)
$$

Once you have $\bar{\alpha}_k$, you compute $\beta_k$ as:
$$
\beta_k = 1 - \frac{\bar{\alpha}_{k+1}}{\bar{\alpha}_k}
$$

Notice that we clip $\beta_k$ to be no larger than $0.999$ in practice to prevent singularities at the end of the diffusion process near $k=K$

In [ ]:
def cosine_beta_schedule(timesteps, s=0.008, dtype=torch.float32):
      """
        cosine noise schedule used in DDPM
      """
      steps = timesteps + 1
      x = np.linspace(0, timesteps, steps)

      # TODO: Compute the value of betas according to the equation
      # Hint:
      # 1. First compute f(k) at each step with the np.cos() function.
      # 2. Compute cumulative product of alphas (alpha_k_bar)
      # 3. Derive betas from alpha_k_bar
      # ================================
      # YOUR CODE GOES HERE





      # ================================

      betas_clipped = np.clip(betas, a_min=0, a_max=0.999)

      return torch.tensor(betas_clipped, dtype=dtype)

### **Verify Your Implementation of the Noise Schedule**

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================
from score import test_cosine_beta_schedule

test_cosine_beta_schedule(cosine_beta_schedule)

## **Q1.2: Complete the Functions in DDPM Class**

In DDPM, we use the cumulative product of $\alpha_k$ and $\beta_k$​ to efficiently express the relationship between the original data $A_0$​ and the noisy sample $A_k$ at any given time step.

We define:
*   $\alpha_k = 1 - \beta_k$, where $\beta_k$ is the noise variance at step $k$.
*   $\bar{\alpha}_k = \prod_{s=1}^{k} \alpha_s$, which represents the cumulative product of all previous $\alpha$ values up to time step $k$.

This gives the relationship between $\bar{\alpha}_k$ and $\beta_k$:
$$
\bar{\alpha}_k = \prod_{s=1}^{k} (1 - \beta_s)
$$

In this way, we could simplify the both the sampling process and the inverse denoising process.

### **1.   Forward diffusion step**
The forward process gradually adds Gaussian noise to the data over $K$ timesteps according to a fixed noise schedule $\beta_k$. At each step $k$, the noisy sample $A_k$ is generated from the previous sample $A_{k-1}$.

By using the cumulative product of $\alpha_k$, instead of iterating through all previous steps to determine the noise level at $A_k$, we use $\bar{\alpha}_k$ to directly express $A_k$ in terms of $A_0$:
$$
A_k = \sqrt{\bar{\alpha}_k} A_0 + \sqrt{1 - \bar{\alpha}_k} \epsilon, \quad \epsilon \sim N(0, I)
$$

This equation is useful because:
*   It allows us to sample $A_k$ in one step, rather than recursively applying the forward process.
*   $\bar{\alpha}_k$ determines how much of the original data $A_0$ remains at step $k$.
*   $1 - \bar{\alpha}_k$ represents the total accumulated noise variance up to step $k$.

### **2.   Backward denoising step**
The backward denoising process aims to recover the original data $A_0$ from $A_K$ by gradually removing the noise.
Since the forward process is Gaussian, the reverse transition is also modeled as a Gaussian distribution:

$$
p(A_{k-1} | A_k) = N(A_{k-1}; \mu, \sigma)
$$

Ho et al. (2020) showed that the mean $\mu$ and variance $\sigma$ of this distribution can be computed in terms of the current sample $A_k$ and the current timestep $k$:
$$
p(A_{k-1} | A_k) = N(\mu(A_k, k), \sigma(k)\mathbf{I}),
$$

where
*   **Mean:** $\mu(A_k, k) := \frac{1}{\sqrt{\alpha_k}} \left( A_k - \frac{\beta_k}{\sqrt{1 - \bar{\alpha}_k}} \, \epsilon_\theta(A_k, k) \right)$  
*   **Variance:** $\sigma(k) := \frac{1 - \bar{\alpha}_{k-1}}{1 - \bar{\alpha}_k} \beta_k$

For a full derivation and explanation, refer to the original paper: (https://arxiv.org/pdf/2006.11239).

In [ ]:
class DDPM(nn.Module):
    def __init__(
        self,
        num_timesteps=100,
        clip_denoised=True,
        output_bound=1.0,
        device='cpu'
    ):
        super(DDPM, self).__init__()

        # denoising timestep
        self.num_timesteps = num_timesteps

        # clip output in each denoising step
        self.clip_denoised = clip_denoised
        self.output_bound = output_bound
        if self.output_bound is None:
            assert not self.clip_denoised, \
            "Cannot clip denoised output if output bound is not set"

        # TODO: Initialize the value for the alpha_k and beta_k:
        # 1, beta_k (using the cosine_beta_schedule that you have created)
        # 2. alpha_k
        # 3. cumulative product of alpha_k
        # ================================
        # YOUR CODE GOES HERE

        # βₖ
        # self.beta_k = ...
        # αₖ = 1 - βₖ
        # self.alpha_k = ...
        # α̅ₖ = ∏ₛ₌₁ᴷ αₛ
        # self.alpha_k_cumprod = ...

         # ================================

        # α̅ₖ₋₁
        self.alpha_k_cumprod_prev = torch.cat(
            [torch.ones(1).to(device), self.alpha_k_cumprod[:-1]]
        )

    def diffuse(self, a_start, k, noise):
        """
            Forward diffusion process

            Args:
                a_start: original clean input sample (a₀)
                k: randomly sample timestep
                noise: random Gaussian noise to be added

            Returns:
                a_k: the noisy version of sample at timestep k.
        """
        self.alpha_k_cumprod = self.alpha_k_cumprod.to(device=a_start.device)
        alpha_k_cumprod = self.alpha_k_cumprod.to(dtype=a_start.dtype)
        k = k.to(a_start.device)
        # TODO: compute the noisy sample Aₖ
        # Hints:
        # 1. Compute values of the two coefficients
        #    a. sqrt_alpha_prod (√ α̅ₖ)
        #    b. sqrt_one_minus_alpha_prod (√ (1-α̅ₖ))
        # Note: Both coefficients must be broadcast to a_start's shape.
        # 2. Compute Aₜ
        #    Aₖ = √ α̅ₖ A₀ + √(1 - α̅ₖ) ε
        # ================================
        # YOUR CODE GOES HERE


        # Aₖ = √ α̅ₖ A₀ + √(1 - α̅ₖ) ε


         # ================================

        return a_k

    def denoise(self, pred_epsilon, k, a_k):
        """
            Backward denoising process

            Args:
                pred_epsilon: predicted noise at timestep k
                k: current timestep
                a_k: noisy input sample a_k at timestep k

            Returns:
                pred_next_sample: predicted sample a_{k-1}
        """
        # TODO: compute the mean and variance of the denoised sample (Aₖ₋₁)
        # Hints:
        # 1. μₖ = (1 / √αₖ) (Aₖ - (βₖ / √(1 - α̅ₖ)) ε)
        # 2. σₖ = βₜ (1-α̅ₖ₋₁) / (1-α̅ₖ)
        # ================================
        # YOUR CODE GOES HERE


        # compute the mean of the predicted denoised sample
        # μₖ = (1 / √αₖ) (Aₖ - (βₖ / √(1 - α̅ₖ)) ε)
        # compute the variance of the predicted denoised sample
        # σₖ = βₜ (1-α̅ₖ₋₁) / (1-α̅ₖ)


         # ================================

        # for k > 0, we add noise to the predicted sample
        # for k = 0, we use deterministic sample
        noise = 0
        if k > 0:
            # clamp the variance to ensure it is not 0
            var_k = torch.clamp(var_k, min=1e-20)
            noise = torch.sqrt(var_k) * torch.randn_like(a_k, device=a_k.device)

        pred_next_sample = mu_k + noise


        return pred_next_sample

## **Test the Diffusion Agent**
Check the data flow of your DDPM. **You can modify it, but theoretically, there is no need to do so.**

In [ ]:
# observation and action dimensions corrsponding to
# the output of PushTEnv
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
obs_dim = 5
action_dim = 2

# create network object
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    diffusion_step_embed_dim=256,
    down_dims=[64, 128, 256],
    kernel_size=5,
    n_groups=4,
)

# device transfer
noise_pred_net = noise_pred_net.to(device)

# example inputs
noised_action = torch.randn((1, pred_horizon, action_dim), device=device)
obs = torch.zeros((1, obs_horizon, obs_dim), device=device)
diffusion_iter = torch.zeros((1,), device=device)

# initialize the noise prediction network
# takes noisy action, diffusion iteration and observation as input
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1)
)

# illustration of removing noise
# the actual noise removal is performed by NoiseScheduler
# and is dependent on the diffusion noise schedule
denoised_action = noised_action - noise

assert noise.shape == (1, pred_horizon, action_dim), \
"Error: The output shape of the noise prediction network was wrong!"
assert denoised_action.shape == (1, pred_horizon, action_dim)
"Error: The shape of the denoised action was wrong!"

## **Q1.3: Complete the Diffusion Policy Training Script**

In this section, you are required to complete the training loop for the diffusion policy.

We have already carefully configured all hyperparameters and training components for you (e.g., network architecture, optimizer, scheduler, EMA, and dataset). Therefore, **you should not modify any part of the script**, except for the section explicitly marked as:

`# YOUR CODE GOES HERE`

---

### Your Task

You need to implement the core training objective of the diffusion policy. Specifically, you should:

- Sample Gaussian noise as the initial diffusion target  
- Randomly sample diffusion timesteps  
- Add noise to the action sequence using the forward diffusion process  
- Predict the noise using the network  
- Compute the MSE loss between predicted noise and ground-truth noise  

---

### Debugging Hint

To help you verify your implementation, we provide a **reference loss curve** (`expected_ddpm_losses`).

During training, your loss curve will be plotted together with the reference curve:

- If your implementation is correct, the two curves should have **similar trends and magnitudes**
- Significant deviation may indicate an error in your implementation

---

### Important Note

- Do **not** change any hyperparameters or training settings  
- Do **not** modify the model architecture or dataset  
- Only complete the missing training logic  

This ensures a fair and consistent evaluation across all submissions.

In [ ]:
#@title {vertical-output: true}

model_path = 'diffusion_policy.pth'
if os.path.exists(model_path):
    print(f"Model already exists at {model_path}, skipping training.")
else:
    # load expected losses for dubugging
    with open('losses/expected_ddpm_loss.pkl', 'rb') as f:
        expected_ddpm_losses = pickle.load(f)

    # set seed (do NOT change!)
    env = PushTEnv()
    env.seed(1)

    # parameters (do NOT change!)
    pred_horizon = 16
    obs_horizon = 2
    action_horizon = 8
    obs_dim = 5
    action_dim = 2

    # create network object (do NOT change!)
    noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
        diffusion_step_embed_dim=256,
        down_dims=[64, 128, 256],
        kernel_size=5,
        n_groups=4,
    )

    # device transfer (do NOT change!)
    noise_pred_net = noise_pred_net.to(device)

    # for this demo, we use DDPMScheduler with 100 diffusion iterations (do NOT change!)
    num_diffusion_iters = 100
    noise_scheduler = DDPM(num_timesteps=num_diffusion_iters)

    # create dataset from file (do NOT change!)
    dataset = PushTStateDataset(
        dataset_path=extract_path,
        pred_horizon=pred_horizon,
        obs_horizon=obs_horizon,
        action_horizon=action_horizon
    )
    # save training data statistics (min, max) for each dim (do NOT change!)
    stats = dataset.stats

    # create dataloader (do NOT change!)
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=256,
        num_workers=1,
        shuffle=True,
        # accelerate cpu-gpu transfer
        pin_memory=True,
        # don't kill worker process afte each epoch
        persistent_workers=True
    )

    # Training epochs (do NOT change!)
    num_epochs = 50

    # (EMA) Exponential Moving Average (do NOT change!)
    # helps accelerate training and improve stability
    ema = EMAModel(
        parameters=noise_pred_net.parameters(),
        power=0.75
    )

    # Standard ADAM optimizer (do NOT change!)
    # Note that EMA parametesr are not optimized
    optimizer = torch.optim.AdamW(
        params=noise_pred_net.parameters(),
        lr=3e-4, weight_decay=1e-6
    )

    # Cosine LR schedule with linear warmup (do NOT change!)
    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=50,
        num_training_steps=len(dataloader) * num_epochs
    )

    # log all epoch losses for plotting
    ddpm_epoch_losses = []

    with tqdm(range(num_epochs), desc='Epoch') as tglobal:
        # epoch loop
        for epoch_idx in tglobal:
            epoch_loss = list()
            # batch loop
            with tqdm(dataloader, desc='Batch', leave=False) as tepoch:
                for nbatch in tepoch:
                    # data normalized in dataset
                    # device transfer
                    nobs = nbatch['obs'].to(device)
                    naction = nbatch['action'].to(device)
                    B = nobs.shape[0]

                    # observation as FiLM conditioning
                    # (B, obs_horizon, obs_dim)
                    obs_cond = nobs[:,:obs_horizon,:]
                    # (B, obs_horizon * obs_dim)
                    obs_cond = obs_cond.flatten(start_dim=1)

                    # TODO: compute the loss for diffusion policy
                    # Hints:
                    # 1. initialize random noise as x_T to get started
                    # 2. use torch.randint() to create random timesteps for
                    #    noise prediction
                    # 3. compute the mse loss between predicted and actual noise (epsilon)
                    # ================================
                    # YOUR CODE GOES HERE








                    # ================================

                    # optimize
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    # step lr scheduler every batch
                    # this is different from standard pytorch behavior
                    lr_scheduler.step()

                    # update Exponential Moving Average of the model weights
                    ema.step(noise_pred_net.parameters())

                    # logging
                    loss_cpu = loss.item()
                    epoch_loss.append(loss_cpu)
                    tepoch.set_postfix(loss=loss_cpu)

            # update progress bar
            tglobal.set_postfix(loss=np.mean(epoch_loss))

            # upate the loss buffer
            mean_loss = np.mean(epoch_loss)
            ddpm_epoch_losses.append(mean_loss)

            # Live update plot
            clear_output(wait=True)
            plt.clf()
            plt.close()
            plt.figure(figsize=(8, 5))
            plt.plot(expected_ddpm_losses, label='Reference')
            plt.plot(ddpm_epoch_losses, linestyle="--", label='Yours')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.title('DDPM Behavioral Cloning Training Loss')
            plt.legend()
            plt.grid(True)
            plt.show()


    # Weights of the EMA model
    # is used for inference
    ema_noise_pred_net = noise_pred_net
    ema.copy_to(ema_noise_pred_net.parameters())
    torch.save(ema_noise_pred_net.state_dict(), model_path)

## **Q1.4: Complete the Diffusion Model Inference Function.**

**Hints:**

1.   Disable Torch gradients during inference.
2.   Perform the reverse denoising process, iterating for *num_diffusion_iters* steps.
2.   Assign the model prediction to the parameter ***naction***, ensuring it has the shape (batch_size, pred_horizon, action_dim).

In [ ]:
# Model Inference

def ddpm_inference(
        ema_model,
        ddpm_scheduler,
        batch_size,
        nobs,
        num_diffusion_iters,
        pred_horizon,
        action_dim,
        device
    ):
    """
      Diffusion agent Inference Function
      Args:
          ema_model: EMA model
          ddpm_scheduler: DDPM noise scheduler
          batch_size: size of the current batch
          nobs: normalized obs
          num_diffusion_iters: DDPM timesteps (100)
          pred_horizon: horizon for predicted actions
          action_dim: action dimension
          device: device

      Returns:
          naction: predicted normalized action sequence
    """
    # TODO: predict action sequence
    # `naction` shape: (batch_size, pred_horizon, action_dim)
    # ================================
    # YOUR CODE GOES HERE



    return action_pred
    # return naction
    # ================================

## **Test and Visualize your policy.**
We provide you with the code to test and visualize your diffusion policy.  **You can modify it, but theoretically, there is no need to do so.**

In [ ]:
#@title {vertical-output: true}
# limit enviornment interaction to 500 steps before termination
max_steps = 500
env = PushTEnv()

# set seed (do NOT change!)
env.seed(1)

# parameters (do NOT change!)
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
obs_dim = 5
action_dim = 2

# create network object (do NOT change!)
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    diffusion_step_embed_dim=256,
    down_dims=[64, 128, 256],
    kernel_size=5,
    n_groups=4,
)

# device transfer (do NOT change!)
noise_pred_net = noise_pred_net.to(device)
noise_pred_net.load_state_dict(torch.load('diffusion_policy.pth'))

# for this demo, we use DDPMScheduler with 100 diffusion iterations (do NOT change!)
num_diffusion_iters = 100
noise_scheduler = DDPM(num_timesteps=num_diffusion_iters)

# create dataset from file (do NOT change!)
dataset = PushTStateDataset(
    dataset_path=extract_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim (do NOT change!)
stats = dataset.stats

# get first observation
obs, info = env.reset()

# keep a queue of last 2 steps of observations
obs_deque = collections.deque(
    [obs] * obs_horizon, maxlen=obs_horizon)
# save visualization and rewards
imgs = [env.render(mode='rgb_array')]
rewards = list()
done = False
step_idx = 0

with tqdm(total=max_steps, desc="Eval PushTStateEnv") as pbar:
    while not done:
        B = 1
        # stack the last obs_horizon (2) number of observations
        obs_seq = np.stack(obs_deque)
        # normalize observation
        nobs = normalize_data(obs_seq, stats=stats['obs'])
        # device transfer
        nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)

        # predict the next action sequence
        naction = ddpm_inference(
            ema_model=noise_pred_net,
            ddpm_scheduler=noise_scheduler,
            batch_size=B,
            nobs=nobs,
            num_diffusion_iters=num_diffusion_iters,
            pred_horizon=pred_horizon,
            action_dim=action_dim,
            device=device,
        )

        # unnormalize action
        naction = naction.detach().to('cpu').numpy()
        # (B, pred_horizon, action_dim)
        naction = naction[0]
        action_pred = unnormalize_data(naction, stats=stats['action'])

        # only take action_horizon number of actions
        start = obs_horizon - 1
        end = start + action_horizon
        action = action_pred[start:end,:]
        # (action_horizon, action_dim)

        # execute action_horizon number of steps
        # without replanning
        for i in range(len(action)):
            # stepping env
            obs, reward, done, _, info = env.step(action[i])
            # save observations
            obs_deque.append(obs)
            #and reward/vis
            rewards.append(reward)
            imgs.append(env.render(mode='rgb_array'))

            # update progress bar
            step_idx += 1
            pbar.update(1)
            pbar.set_postfix(reward=reward)
            if step_idx > max_steps:
                done = True
            if done:
                break

# print out the maximum target coverage
print('Score: ', max(rewards))

# visualize
from IPython.display import Video
vwrite('ddpm_vis.mp4', imgs)
Video('ddpm_vis.mp4', embed=True, width=256, height=256)

## **Grading and Evaluation for Part 1**

Your trained policy will be evaluated over **10 independent episodes**, each consisting of a **500-steps rollout**.

In each episode:

- The environment is initialized with a different random seed.
- Your controller generates actions using the learned policy.
- The **Maximum** reward (percentage of coverage) over the episode is recorded.

---
### Final Score

After all 10 episodes:

- The **average penalized reward** is computed.
- Your final score is determined based on the following thresholds:

  - Average reward ≥ 0.85 → **2 points**
  - Average reward ≥ 0.6 → **1 point**
  - Average reward < 0.6 → **0 points**

---

This evaluation emphasizes not only performance but also **robustness across different initial conditions**. Controllers that are unstable or sensitive to distribution shifts will receive lower scores.

You are **not allowed to modify any other code in the evaluation cell**.

In [ ]:
#@title {vertical-output: true}
# ================================
# DO NOT MODIFY THIS CELL
# ================================
from util import save_video_with_cv2

# limit enviornment interaction to 500 steps before termination
max_steps = 500
env = PushTEnv()
# set seed (do NOT change!)
env.seed(1)
# get first observation
obs, info = env.reset()

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
obs_dim = 5
action_dim = 2

# create network object
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    diffusion_step_embed_dim=256,
    down_dims=[64, 128, 256],
    kernel_size=5,
    n_groups=4,
)

# create dataset from file (do NOT change!)
dataset = PushTStateDataset(
    dataset_path=extract_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim (do NOT change!)
stats = dataset.stats

# device transfer
noise_pred_net = noise_pred_net.to(device)
noise_pred_net.load_state_dict(torch.load('diffusion_policy.pth'))

# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100
noise_scheduler = DDPM(num_timesteps=num_diffusion_iters)


score_list = list()
combined_imgs = [env.render(mode='rgb_array')]

for iter in range(10):
    env.seed(iter)

    # get first observation
    obs, info = env.reset()

    # keep a queue of last 2 steps of observations
    obs_deque = collections.deque(
        [obs] * obs_horizon, maxlen=obs_horizon)
    # save visualization and rewards
    rewards = list()
    done = False
    step_idx = 0

    with tqdm(total=max_steps, desc=f"Eval PushTStateEnv episode:{iter}") as pbar:
        while not done:
            B = 1
            # stack the last obs_horizon (2) number of observations
            obs_seq = np.stack(obs_deque)
            # normalize observation
            nobs = normalize_data(obs_seq, stats=stats['obs'])
            # device transfer
            nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)

            # predict the next action sequence
            naction = ddpm_inference(
                ema_model=noise_pred_net,
                ddpm_scheduler=noise_scheduler,
                batch_size=B,
                nobs=nobs,
                num_diffusion_iters=num_diffusion_iters,
                pred_horizon=pred_horizon,
                action_dim=action_dim,
                device=device,
            )

            # unnormalize action
            naction = naction.detach().to('cpu').numpy()
            # (B, pred_horizon, action_dim)
            naction = naction[0]
            action_pred = unnormalize_data(naction, stats=stats['action'])

            # only take action_horizon number of actions
            start = obs_horizon - 1
            end = start + action_horizon
            action = action_pred[start:end, :]
            # (action_horizon, action_dim)

            # execute action_horizon number of steps
            # without replanning
            for i in range(len(action)):
                # stepping env
                obs, reward, done, _, info = env.step(action[i])
                # save observations
                obs_deque.append(obs)
                # and reward/vis
                rewards.append(reward)
                combined_imgs.append(env.render(mode='rgb_array'))

                # update progress bar
                step_idx += 1
                pbar.update(1)
                pbar.set_postfix(reward=reward)
                if step_idx > max_steps:
                    done = True
                if done:
                    break

    # print out the maximum target coverage
    print('Score: ', max(rewards))
    # add to the score list
    score_list.append(max(rewards))

print(f"Average Reward: {np.mean(score_list)}")
# compute the mean of the scores
# Reward score
if np.mean(score_list) >= 0.85:
    final_score_part1 = 2
elif np.mean(score_list) >= 0.6:
    final_score_part1 = 1
else:
    final_score_part1 = 0

print(f"Score for Part1: {final_score_part1}/2")

# visualize
print("Video saved successfully!")
from IPython.display import Video
vwrite('ddpm_combined_vis.mp4', combined_imgs)
Video('ddpm_combined_vis.mp4', embed=True, width=256, height=256)

# **Part 2. Reinforcement Learning with an Open-Source RL Library**

In the previous sections, you worked with imitation learning methods on the PushT task. However, this environment has a **very narrow and constrained action space**, and meaningful rewards are only obtained when the actions are close to optimal.

As a result:

- Random exploration rarely produces useful behavior  
- Rewards are **extremely sparse**  
- Reinforcement learning algorithms struggle to receive meaningful learning signals during early training  

This makes PushT **unsuitable for standard reinforcement learning**, especially without careful reward shaping or advanced exploration strategies.

---

## Why We Switch to HalfCheetah

To better study reinforcement learning, we return to the **Half-Cheetah** task:

- Rewards are **dense and continuous**  
- Random actions can still produce non-zero rewards  
- Learning signals are more stable and informative  

This makes it a more appropriate benchmark for training RL agents.

---

## Stable-Baselines3

In this part, we will use the open-source library to train reinforcement learning agents.



## **Vital Code Explanation for Cheetah Environment**
We have already:

- Wrapped the `dm_control` environment into a Gym-compatible interface  
- Adapted it for vectorized training  
- Configured the environment for use with Stable-Baselines3  

---

## Important Note

You are **not allowed to modify any code** in the environment wrapper (`ENV_for_RL`) or the vectorized environment setup.

This ensures:

- Compatibility with the RL library  
- Fair and consistent evaluation across all submissions  

You should focus only on implementing and training your RL agent using the provided interface.

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================

class ENV_for_RL(gym.Env):
    metadata = {"render_modes": ["rgb_array"], "render_fps": 25}
    def __init__(self, env):
        super().__init__()
        self._env = env

        self.MPC_frequency = 25
        self.control_timestep = 1.0 / self.MPC_frequency
        self.MPC_repeats = max(1, int(round(self.control_timestep / env.control_timestep())))
        self.max_episode_steps = 100

        # --- Build observation_space from dm_control's observation spec ---
        obs_spec = env.observation_spec()
        obs_dim = sum(
            int(np.prod(spec.shape)) if spec.shape else 1
            for spec in obs_spec.values()
        )
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        # --- Build action_space from dm_control's action spec ---
        action_spec = env.action_spec()
        self.action_space = spaces.Box(
            low=action_spec.minimum.astype(np.float32),
            high=action_spec.maximum.astype(np.float32),
            dtype=np.float32,
        )

        self.clear()

    def _flatten_obs(self, obs_dict) -> np.ndarray:
        """Flatten dm_control's OrderedDict observation into a 1-D array."""
        return np.concatenate([
            np.atleast_1d(v).ravel() for v in obs_dict.values()
        ]).astype(np.float32)

    def clear(self):
        self.frames = []
        self.qposs = []
        self.qvels = []
        self.step_num = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.clear()
        time_step = self._env.reset()
        obs = self._flatten_obs(time_step.observation)
        return obs, {}

    def step(self, action, record: bool = False):
        total_reward = 0.0

        for _ in range(self.MPC_repeats):
            time_step = self._env.step(action)
            total_reward += time_step.reward or 0.0
            if time_step.last():
                break

        obs = self._flatten_obs(time_step.observation)
        terminated = time_step.last()
        self.step_num += 1
        truncated = self.step_num >= self.max_episode_steps
        info = {'raw_reward':total_reward}

        if record:
            self._record_frame()

        return obs, total_reward, terminated, truncated, info

    def render(self):
        return self._env.physics.render(camera_id=0, width=256, height=256)

    def close(self):
        self._env.close()

    def get_state(self):
        return self._env.physics.get_state()

    def set_state(self, state):
        self._env.physics.set_state(state)
        self._env.physics.forward()

    def _record_frame(self):
        self.frames.append(self._env.physics.render(camera_id=0, width=256, height=256))
        self.qposs.append(copy.deepcopy(self._env.physics.data.qpos))
        self.qvels.append(copy.deepcopy(self._env.physics.data.qvel))

    def get_log(self):
        return self.frames, self.qposs, self.qvels


## **Training the RL Agent**

In this section, you will train a reinforcement learning agent using the
Stable-Baselines3 framework.

We recommend using **PPO (Proximal Policy Optimization)**, as it is a mature and stable on-policy algorithm. However, you are also free to experiment with other algorithms such as SAC or TD3.

---

### Your Task

You need to:

- Instantiate an RL model (e.g., PPO)  
- Train the model using the provided vectorized environment  
- Save the trained policy  

---

### Parallel Environments

One major advantage of simulation-based RL is the ability to run **multiple environments in parallel**.

This allows:

- Faster data collection  
- More stable training  
- Better utilization of CPU resources  

We provide a helper function `make_vec_env` to create parallel environments.

- We recommend using **8 environments** as a good balance between speed and Colab CPU limits  
- You may experiment with different values to explore performance trade-offs  

---

### Hyperparameters

Each RL algorithm has many hyperparameters that can significantly affect performance. You are encouraged to experiment with:

- `total_timesteps` (training duration)  
- `batch_size`  
- Network architecture (`net_arch`)  
- Learning rate and other RL-specific parameters  

Tuning these parameters can lead to noticeable improvements in performance and training efficiency.

---

### Important Note

- You should only modify the section marked as `YOUR CODE GOES HERE`  
- Do not change the environment setup or wrapper  
- Make sure to save your trained model for evaluation  

This part is designed to give you hands-on experience with practical RL training pipelines.

In [ ]:
#@title {vertical-output: true}
from stable_baselines3.ppo import PPO

class EnvMaker:
    def __init__(self, seed):
        self.seed = seed

    def __call__(self):
        random_state = np.random.RandomState(self.seed)
        env = ENV_for_RL(suite.load('cheetah', 'run', task_kwargs={'random': random_state}))
        return env


def make_vec_env(nenv, seed):
    venv = VecMonitor(SubprocVecEnv([EnvMaker(seed + 100 * i) for i in range(nenv)]))
    return venv

model_path = 'rl_policy'
if os.path.exists(model_path+'.zip'):
    print(f"Model already exists at {model_path}, skipping training.")
else:
    # ================================
    # YOUR CODE GOES HERE

    # Default parameters
    total_timesteps = 500000
    nenv = 8  # number of parallel environments. This can speed up training when you have good CPUs
    seed = 8

    # Set random seed
    set_random_seed(seed)

    # Create parallel envs
    vec_env = make_vec_env(nenv=nenv, seed=seed)



    # ================================

## **Manual Testing and Visualization**

Same as previous assignments, we provide a visualization script to help you evaluate the performance of your trained policy.

The script runs a rollout in the environment, records rewards, states, and rendered frames, and visualizes them for analysis.

---


### Important Note

To reduce grading time, **please comment out the entire testing/visualization block in your final submission**.


In [ ]:
#@title {vertical-output: true}
import copy
import matplotlib.pyplot as plt

random_state = np.random.RandomState(210)
np.random.seed(210)
env = ENV_for_RL(suite.load('cheetah', 'run', task_kwargs={'random': random_state}))

# Set simulation duration and calculate number of steps
duration = 4.0  # seconds
num_steps = int(duration / env.control_timestep)


# ================================
# YOUR CODE GOES HERE ONLY
model = PPO.load("rl_policy.zip", env=env)

# ================================

rewards = []
ticks = []
action_sequence = []
obs, _ = env.reset()
active_force_until = -1
spec = suite.load('cheetah', 'run', task_kwargs={'random': random_state}).action_spec()
# Main simulation loop
for step in range(num_steps):



    # Plan: Get best action sequence via MPC and take first action
    current_state = env.get_state()

    action, _ = model.predict(obs, deterministic=True)

    # Execute: Apply action to environment
    obs, r, _, _, info = env.step(action,True)

    reward = info['raw_reward']

    # Record: Collect data from this step
    rewards.append(reward)
    ticks.append(step)

    # Print progress every 10 steps
    if step % 10 == 0:
        print(f"Step {step}/{num_steps}, total reward: {sum(rewards):.2f}, avg reward: {np.mean(rewards):.4f}")

frames, qposs, qvels = env.get_log()
print(f"{len(frames)} frames, total reward: {sum(rewards):.2f}, avg reward: {np.mean(rewards):.4f}")
visualize_log(ticks, rewards, frames, qposs, qvels)

env.close()
# Display video of the cheetah running
show_video(frames, fps=25)

## **Grading and Evaluation for Part 2**

Your trained policy will be evaluated over **100 independent episodes**, each consisting of a **4-second rollout**.

In each episode:

- The environment is initialized with a different random seed.
- Your controller generates actions using the learned policy.
- The accumulated reward over the episode is recorded.

---

### Stability Penalty

If the cheetah falls during an episode (i.e., the torso pitch angle exceeds the threshold):

- The reward for that episode will be **penalized**.
- The number of such failures will also be recorded.

---

### Final Score

After all 100 episodes:

- The **average penalized reward** is computed.
- Your final score is determined based on the following thresholds:
  - Average reward ≥ 100 → **3 points**
  - Average reward ≥ 80 → **2 points**
  - Average reward ≥ 40 → **1 point**
  - Average reward < 40 → **0 points**

---

This evaluation emphasizes not only performance but also **robustness across different initial conditions**. Controllers that are unstable or sensitive to distribution shifts will receive lower scores.

Except for the line:

    model = PPO.load("ppo_network.zip", env=env)

you are **not allowed to modify any other code in the evaluation cell**.

In [ ]:
#@title{vertical-output: true}
import copy
import time
import matplotlib.pyplot as plt


# Set simulation duration and calculate number of steps
duration = 4.0  # seconds

# Instantiate your Agent controller
# but do NOT modify anything else in this cell.
# ================================
# YOUR CODE GOES HERE ONLY
model = PPO.load("rl_policy.zip", env=env)
# ================================

rewards = []
flipped_count = 0

for i in range(100):

  random_state = np.random.RandomState(i+200)
  np.random.seed(i)
  env = ENV_for_RL(suite.load('cheetah', 'run', task_kwargs={'random': random_state}))

  obs, _ = env.reset()

  num_steps = int(duration / env.control_timestep)
  reward = 0.0
  flipped = False

  for step in range(num_steps):
      current_state = env.get_state()

      # ---- Fall detection (pitch angle) ----
      if abs(current_state[2]) > np.pi / 2:
          flipped = True

      action, _ = model.predict(obs, deterministic=True)
      obs, _, _, _, info = env.step(action)


      r = info['raw_reward']

      reward += r

  env.close()
  if flipped:
    reward = reward/2.0
    flipped_count += 1
  else:
    reward = reward
  print(f"Episode {i+1} reward: {reward:.2f}", flipped)
  rewards.append(reward)

avg_reward = np.mean(rewards)
print(f"Average penalized reward: {avg_reward:.2f} with {str(flipped_count)} times flipping")

# Reward score
if avg_reward >= 100:
    final_score_part2 = 3
elif avg_reward >= 80:
    final_score_part2 = 2
elif avg_reward >= 40:
    final_score_part2 = 1
else:
    final_score_part2 = 0

print(f"Score for Part2: {final_score_part2}/3")

# **Total Score Calculation**

In [ ]:
# ================================
# DO NOT MODIFY THIS CELL
# ================================
print(f"Final Score: {final_score_part1 + final_score_part2}/5")